# Experiment 02: Batch Endpoint for VLM

**Goal**: Deploy a VLM as an Azure ML Batch Endpoint and run a scoring job.

**Source**: https://learn.microsoft.com/en-us/azure/machine-learning/how-to-use-batch-model-deployments

**Expected duration**: ~45 minutes (including 15 min deployment + 10 min job run)

**Cost**: ~$1.50 (A100 for ~25 min job)

**Key difference from online endpoints**:
- Batch endpoints DON'T use HuggingFace registry models directly.
- We need a **scoring script** that loads the model and processes mini-batches.
- Input is files in Azure Storage (not HTTP payloads).
- Output is a file in Azure Storage (predictions.csv).
- Compute spins up for the job and spins down after — pay only for job time.

## What this notebook proves
- Compute cluster with A100 GPU works
- Batch endpoint + deployment can be created
- Scoring script can load a VLM and run inference on batch images
- Output is accessible in Azure Storage
- We can compare cost/latency with online endpoint

## 1. Setup & Authentication

In [ ]:
# Install dependencies (uncomment if needed)
# !pip install azure-ai-ml azure-identity python-dotenv --upgrade --quiet

In [ ]:
import os
import json
import time
from uuid import uuid4
from dotenv import load_dotenv

load_dotenv()

from azure.ai.ml import MLClient, Input
from azure.ai.ml.entities import (
    BatchEndpoint,
    ModelBatchDeployment,
    ModelBatchDeploymentSettings,
    Model,
    AmlCompute,
    CodeConfiguration,
    Environment,
    BatchRetrySettings,
)
from azure.ai.ml.constants import AssetTypes, BatchDeploymentOutputAction
from azure.identity import DefaultAzureCredential

print("All imports OK")

In [ ]:
# Configure
SUBSCRIPTION_ID = os.getenv("SUBSCRIPTION_ID", "<your-subscription-id>")
RESOURCE_GROUP = os.getenv("RESOURCE_GROUP", "<your-resource-group>")
WORKSPACE_NAME = os.getenv("WORKSPACE_NAME", "<your-workspace-name>")

ENDPOINT_NAME = f"bogota-batch-{str(uuid4())[:8]}"
DEPLOYMENT_NAME = f"qwen-vl-batch-{str(uuid4())[:8]}"
COMPUTE_NAME = "gpu-a100-cluster"

# Authenticate
credential = DefaultAzureCredential()

client = MLClient(
    credential=credential,
    subscription_id=SUBSCRIPTION_ID,
    resource_group_name=RESOURCE_GROUP,
    workspace_name=WORKSPACE_NAME,
)

ws = client.workspaces.get(WORKSPACE_NAME)
print(f"✅ Connected to workspace: {ws.name}")
print(f"   Endpoint: {ENDPOINT_NAME}")
print(f"   Deployment: {DEPLOYMENT_NAME}")
print(f"   Compute: {COMPUTE_NAME}")

## 2. Create GPU Compute Cluster

Batch endpoints run on compute clusters. We create one with A100 GPU.
The cluster auto-scales: 0 nodes when idle (no cost), up to N when jobs run.

In [ ]:
print(f"Creating compute cluster '{COMPUTE_NAME}'...")

compute_cluster = AmlCompute(
    name=COMPUTE_NAME,
    description="A100 GPU cluster for batch VLM inference",
    size="Standard_NC24ads_A100_v4",  # Your A100-80GB
    min_instances=0,   # Scale to 0 when idle (no cost)
    max_instances=1,   # Max 1 node (you have 1 GPU)
    idle_time_before_scale_down=300,  # 5 min idle before scaling down
)

try:
    client.compute.begin_create_or_update(compute_cluster).wait()
    print(f"✅ Compute cluster '{COMPUTE_NAME}' created")
except Exception as e:
    if "already exists" in str(e).lower():
        print(f"⚠ Cluster already exists, reusing it")
    else:
        print(f"❌ Failed: {e}")
        print("\nCheck GPU quota for Standard_NC24ads_A100_v4 in your region")
        raise

## 3. Create Scoring Script

Batch endpoints need a scoring script with `init()` and `run()` functions.
This script loads a VLM and processes batches of images.

Since we can't use the HuggingFace registry for batch (different mechanism),
we register a model that points to our scoring code. The scoring script itself
will call our already-deployed ONLINE endpoint for inference.

This is a practical approach: use the batch endpoint for orchestration
(parallel file processing, retry, job tracking) while the actual VLM inference
happens on the online endpoint.

In [ ]:
import os

# Create scoring code directory
os.makedirs("batch_scoring/code", exist_ok=True)

scoring_script = '''
import os
import json
import time
import pandas as pd
from typing import List
from openai import OpenAI
from urllib.request import urlretrieve
from pathlib import Path


def init():
    """Initialize: set up OpenAI client to call online endpoint."""
    global openai_client, model_name, deployment_name
    
    api_url = os.environ.get("ONLINE_API_URL", "")
    api_key = os.environ.get("ONLINE_API_KEY", "")
    model_name = os.environ.get("MODEL_NAME", "Qwen/Qwen2.5-VL-32B-Instruct")
    deployment_name = os.environ.get("ONLINE_DEPLOYMENT", "")
    
    if not api_url or not api_key:
        raise ValueError("ONLINE_API_URL and ONLINE_API_KEY must be set")
    
    openai_client = OpenAI(
        base_url=f"{api_url}/v1",
        api_key=api_key,
        default_headers={"azureml-model-deployment": deployment_name},
    )
    
    print(f"Batch scorer initialized — calling online endpoint at {api_url}")


PROMPT_TEMPLATE = (
    "Classify this building image from Bogota, Colombia. "
    "Return a JSON object with: "
    '"label": one of [RESIDENCIAL_1, COMERCIAL_1, COMERCIAL_2, COMERCIAL_3, '
    'DOTACIONAL_1, DOTACIONAL_2, MOLES_1, RURAL_1, MIXTO_1, MIXTO_2, UNKNOWN], '
    '"confidence": 0-1. Only return the JSON.'
)


def run(mini_batch: List[str]) -> pd.DataFrame:
    """Process a mini-batch of images."""
    print(f"Processing batch of {len(mini_batch)} images...")
    results = []
    
    for image_path in mini_batch:
        try:
            # Read image and convert to base64 data URL
            import base64
            with open(image_path, "rb") as f:
                image_data = base64.b64encode(f.read()).decode("utf-8")
            
            # Determine MIME type from extension
            ext = Path(image_path).suffix.lower()
            mime = {".jpg": "jpeg", ".jpeg": "jpeg", ".png": "png", ".webp": "webp"}.get(ext, "jpeg")
            data_url = f"data:image/{mime};base64,{image_data}"
            
            # Call online endpoint
            response = openai_client.chat.completions.create(
                model=model_name,
                messages=[
                    {
                        "role": "user",
                        "content": [
                            {"type": "text", "text": PROMPT_TEMPLATE},
                            {"type": "image_url", "image_url": {"url": data_url}},
                        ],
                    }
                ],
                max_tokens=128,
            )
            
            raw = response.choices[0].message.content
            try:
                parsed = json.loads(raw)
                label = parsed.get("label", "PARSE_ERROR")
                confidence = parsed.get("confidence", 0)
            except json.JSONDecodeError:
                label = "PARSE_ERROR"
                confidence = 0
            
            results.append({
                "file": Path(image_path).name,
                "label": label,
                "confidence": confidence,
                "status": "success",
            })
            
        except Exception as e:
            results.append({
                "file": Path(image_path).name if image_path else "unknown",
                "label": "ERROR",
                "confidence": 0,
                "status": f"error: {str(e)[:100]}",
            })
    
    return pd.DataFrame(results)
'''

with open("batch_scoring/code/batch_driver.py", "w") as f:
    f.write(scoring_script)

print("✅ Scoring script created at batch_scoring/code/batch_driver.py")

## 4. Create Environment

The batch deployment needs Python dependencies. We use a conda environment
with `openai`, `pandas`, and the required Azure ML packages.

In [ ]:
import os
os.makedirs("batch_scoring/environment", exist_ok=True)

conda_yaml = '''
name: batch-vlm-env
channels:
  - conda-forge
dependencies:
  - python=3.10
  - pip
  - pip:
    - openai>=1.0.0
    - pandas>=2.0.0
    - azureml-core
    - azureml-dataset-runtime[fuse]
'''

with open("batch_scoring/environment/conda.yaml", "w") as f:
    f.write(conda_yaml)

print("✅ Environment conda.yaml created")

# Create environment in Azure ML
env = Environment(
    name="batch-vlm-env",
    conda_file="batch_scoring/environment/conda.yaml",
    image="mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu22.04:latest",
)

try:
    client.environments.create_or_update(env)
    print("✅ Environment registered in Azure ML")
except Exception as e:
    print(f"⚠ Environment registration: {e}")
    print("   Will use inline environment in deployment")

## 5. Register a Placeholder Model

Batch deployments require a registered model. Since our scoring script calls
the online endpoint, the model isn't actually loaded locally. We register a
placeholder.

In [ ]:
# Create placeholder model file
os.makedirs("batch_scoring/model", exist_ok=True)
with open("batch_scoring/model/model_info.json", "w") as f:
    json.dump({
        "model_type": "vlm-batch-router",
        "description": "Routes batch requests to online VLM endpoint",
        "online_endpoint": os.getenv("EXP_ONLINE_ENDPOINT", ""),
    }, f, indent=2)

# Register in Azure ML
try:
    model = client.models.create_or_update(
        Model(
            name="bogota-vlm-batch-router",
            path="batch_scoring/model/",
            type=AssetTypes.CUSTOM_MODEL,
            description="Batch router to online VLM endpoint",
        )
    )
    print("✅ Placeholder model registered")
except Exception as e:
    print(f"⚠ Model registration: {e}")
    print("   Continuing without explicit model registration")

## 6. Create Batch Endpoint & Deployment

⏱️ **This cells take ~5 minutes for endpoint creation + up to 15 minutes
for deployment to be fully provisioned**

In [ ]:
# Create batch endpoint
print(f"Creating batch endpoint: {ENDPOINT_NAME}...")

endpoint = BatchEndpoint(
    name=ENDPOINT_NAME,
    description="Bogota land-use VLM — experimental batch endpoint",
)

try:
    client.batch_endpoints.begin_create_or_update(endpoint).wait()
    print(f"✅ Batch endpoint created")
except Exception as e:
    print(f"❌ Failed: {e}")
    raise

In [ ]:
# Create deployment
print(f"Creating deployment: {DEPLOYMENT_NAME}...")

# Get online endpoint credentials from notebook 01
online_api_url = os.getenv("EXP_ONLINE_API_URL", "")
online_api_key = os.getenv("EXP_ONLINE_API_KEY", "")
online_deployment = os.getenv("EXP_ONLINE_DEPLOYMENT", "")

if not online_api_url or not online_api_key:
    print("⚠ WARNING: Online endpoint credentials not found in .env")
    print("   Run notebook 01 first, or set EXP_ONLINE_API_URL and EXP_ONLINE_API_KEY")

deployment = ModelBatchDeployment(
    name=DEPLOYMENT_NAME,
    endpoint_name=ENDPOINT_NAME,
    model="bogota-vlm-batch-router:1",  # Our placeholder
    code_configuration=CodeConfiguration(
        code="batch_scoring/code/",
        scoring_script="batch_driver.py",
    ),
    environment=env,
    compute=COMPUTE_NAME,
    instance_count=1,
    settings=ModelBatchDeploymentSettings(
        max_concurrency_per_instance=1,  # 1 at a time for VLM (GPU-bound)
        mini_batch_size=1,               # Process 1 image per run() call
        output_action=BatchDeploymentOutputAction.APPEND_ROW,
        output_file_name="predictions.csv",
        retry_settings=BatchRetrySettings(max_retries=2, timeout=120),
        logging_level="info",
        environment_variables={
            "ONLINE_API_URL": online_api_url,
            "ONLINE_API_KEY": online_api_key,
            "ONLINE_DEPLOYMENT": online_deployment,
            "MODEL_NAME": "Qwen/Qwen2.5-VL-32B-Instruct",
        },
    ),
)

deployment_start = time.time()
try:
    client.batch_deployments.begin_create_or_update(deployment).wait()
    deployment_elapsed = time.time() - deployment_start
    print(f"✅ Batch deployment created in {deployment_elapsed/60:.1f} min")
    
    # Set as default
    endpoint = client.batch_endpoints.get(ENDPOINT_NAME)
    endpoint.defaults.deployment_name = DEPLOYMENT_NAME
    client.batch_endpoints.begin_create_or_update(endpoint).wait()
    print(f"✅ Set as default deployment")
except Exception as e:
    print(f"❌ Deployment failed: {e}")
    print("\nCheck:")
    print("  1. Compute cluster is running")
    print("  2. Environment conda.yaml is valid")
    print("  3. GPU quota for Standard_NC24ads_A100_v4")
    raise

## 7. Prepare Test Data

Batch endpoints read input from Azure Storage. We'll upload a few test images
to the workspace's default blob store.

In [ ]:
# Download a few test images to local
import urllib.request

os.makedirs("batch_scoring/test_data", exist_ok=True)

test_images = [
    ("https://upload.wikimedia.org/wikipedia/commons/thumb/4/4f/Flatiron_Building_3618433845_5745e68717.jpg/800px-Flatiron_Building_3618433845_5745e68717.jpg", "flatiron.jpg"),
    ("https://upload.wikimedia.org/wikipedia/commons/thumb/5/5e/Upper_East_Side_apartment_buildings.jpg/800px-Upper_East_Side_apartment_buildings.jpg", "apartments.jpg"),
    ("https://upload.wikimedia.org/wikipedia/commons/thumb/8/8b/Brownstones_at_112-114_East_74th_Street.jpg/800px-Brownstones_at_112-114_East_74th_Street.jpg", "brownstones.jpg"),
]

for url, filename in test_images:
    dest = f"batch_scoring/test_data/{filename}"
    urllib.request.urlretrieve(url, dest)
    print(f"Downloaded: {filename}")

print(f"\n✅ {len(test_images)} test images downloaded")

In [ ]:
# Register test data as an Azure ML data asset
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

try:
    data_asset = client.data.create_or_update(
        Data(
            name="bogota_batch_test_images",
            path="batch_scoring/test_data/",
            type=AssetTypes.URI_FOLDER,
            description="Test images for batch VLM classification",
        )
    )
    print(f"✅ Test data registered: {data_asset.path}")
    TEST_DATA_PATH = data_asset.path
except Exception as e:
    print(f"⚠ Data registration failed: {e}")
    print("   Trying direct path upload...")
    TEST_DATA_PATH = "batch_scoring/test_data/"

## 8. Invoke Batch Endpoint

Submit a batch scoring job. This is the actual work.

⏱️ **This takes 5-15 minutes depending on cluster startup + inference time.**

In [ ]:
print(f"Invoking batch endpoint {ENDPOINT_NAME}...")
print(f"  Deployment: {DEPLOYMENT_NAME}")
print(f"  Input: {TEST_DATA_PATH}")

job_start = time.time()

try:
    job = client.batch_endpoints.invoke(
        endpoint_name=ENDPOINT_NAME,
        deployment_name=DEPLOYMENT_NAME,
        input=Input(
            path=TEST_DATA_PATH,
            type=AssetTypes.URI_FOLDER,
        ),
    )
    
    print(f"✅ Job submitted: {job.name}")
    print(f"   Monitor at: https://ml.azure.com")
    JOB_NAME = job.name
    
except Exception as e:
    print(f"❌ Job submission failed: {e}")
    print("\nEnsure the online endpoint from notebook 01 is running!")
    print("Batch endpoint calls the online endpoint for inference.")
    raise

## 9. Monitor Job Progress

In [ ]:
print("Waiting for job to complete...")

max_wait = 600  # 10 minutes timeout
poll_interval = 15  # seconds

for i in range(max_wait // poll_interval):
    time.sleep(poll_interval)
    job_detail = client.jobs.get(JOB_NAME)
    status = job_detail.status
    
    elapsed = time.time() - job_start
    print(f"  [{elapsed:.0f}s] Status: {status}")
    
    if status in ("Completed", "Failed", "Canceled"):
        break
else:
    print("⚠ Job timed out — check Azure ML Studio for status")

job_elapsed = time.time() - job_start
print(f"\nJob finished in {job_elapsed:.0f}s with status: {status}")

## 10. Access Results

Batch job output is stored in Azure Storage.

In [ ]:
print(f"Job {JOB_NAME} completed with status: {status}")

# Get job details with outputs
job_detail = client.jobs.get(JOB_NAME)

print(f"\nJob details:")
print(f"  Status: {job_detail.status}")
print(f"  Outputs: {job_detail.outputs if hasattr(job_detail, 'outputs') else 'N/A'}")

# Open Azure ML Studio to view results
studio_url = f"https://ml.azure.com/job/{JOB_NAME}?wsid=/subscriptions/{SUBSCRIPTION_ID}/resourceGroups/{RESOURCE_GROUP}/providers/Microsoft.MachineLearningServices/workspaces/{WORKSPACE_NAME}"
print(f"\n📊 View results in Azure ML Studio:")
print(f"   {studio_url}")
print(f"\n   In studio: Job → Outputs + logs → Select batchscoring → Show data outputs")
print(f"   The predictions.csv file contains the classification results.")

## 11. Batch Summary

In [ ]:
# Cost estimate
A100_COST_PER_HOUR = 3.60
job_cost = (job_elapsed / 3600) * A100_COST_PER_HOUR

print("=" * 50)
print("BATCH ENDPOINT RESULTS")
print("=" * 50)
print(f"Model:            Qwen2.5-VL-32B-Instruct (via online endpoint)")
print(f"Compute:          {COMPUTE_NAME}")
print(f"Deployment time:  {deployment_elapsed/60:.1f} min")
print(f"Job duration:     {job_elapsed:.0f}s ({job_elapsed/60:.1f} min)")
print(f"Images processed: {len(test_images)}")
print(f"Avg per image:    {job_elapsed/len(test_images):.1f}s")
print(f"Job cost:         ${job_cost:.4f}")
print(f"")
print(f"⚠ Batch endpoint has ZERO cost when idle (scale-to-zero cluster).")
print(f"   You pay only for job execution time.")
print(f"   Cluster cold start adds 2-4 min to first job.")

## 12. Clean Up

In [ ]:
# Delete batch endpoint
print(f"Deleting batch endpoint {ENDPOINT_NAME}...")
try:
    client.batch_endpoints.begin_delete(name=ENDPOINT_NAME).result()
    print("✅ Batch endpoint deleted")
except Exception as e:
    print(f"⚠ Could not delete: {e}")

# The compute cluster auto-scales to 0, so no ongoing cost.
# Keep the cluster for future batch jobs, or delete it:
# client.compute.begin_delete(name=COMPUTE_NAME).result()
print("✅ Compute cluster will auto-scale to 0 (no cost when idle)")

## 13. Online vs Batch Comparison

Fill in from your experimental results:

| Criterion | Online Endpoint | Batch Endpoint |
|-----------|----------------|----------------|
| Deployment time | ____ min | ____ min |
| Single image | ____ sec | ____ sec |
| Multi-image (4) | ____ sec | N/A |
| Cost model | $3.60/hr running | $3.60/hr only during job |
| Scale to zero | Yes (cold start) | Yes (auto, per-job) |
| Max inputs/call | HTTP payload limit | Unlimited (Azure Storage) |
| Best for | Demo, review UI | 1,000+ building jobs |
| Complexity | Low (OpenAI SDK) | Medium (scoring script) |

## Recommendation

Based on these results:
- **For MVP**: Use online endpoint for everything (demo + batch via async processing).
  Simpler, cheaper at <1000 buildings.
- **For production scale**: Consider batch endpoint if jobs exceed 500 buildings
  or require parallel processing across multiple GPUs.

Return to `DECISION_GATE_PLAN.md` and proceed to Phase 02.